Programatic solutions to 2 of the 3 big problems in HMMs. Credit to Rabiner as well as Jurafsky and Martin. 

Also an implementation of the occasionally dishonest casino example from Durbin's Biological Sequence Analysis.

In [198]:
import numpy as np

Initialising ice-cream Markov chain found in "Speech and Language Processing" by Jurafsky and Martin.

In [199]:
# cold - hot
A = np.array([[0.5, 0.5], [0.4, 0.6]])
print(A)
B = np.array([[0.5, 0.4, 0.1], [0.2, 0.4, 0.4]])
print(B)
pi = [0.2, 0.8]
O=[2,0,2]

[[0.5 0.5]
 [0.4 0.6]]
[[0.5 0.4 0.1]
 [0.2 0.4 0.4]]


In [200]:
A @ B
c = np.array([2, 1, 0])
c * B

array([[1. , 0.4, 0. ],
       [0.4, 0.4, 0. ]])

# The Forward Algorithm
A dynamic programming solution to the problem:

Given a Hiden Markov Model $\lambda = (A,B)$ and an observation sequence $O$, determine the likelihood $P(O|\lambda)$.

In [201]:
def forward(A, B, pi, O):
    """
    A: ndarray(N, N) state transition probabilities
    B: ndarray(N, M) state observation probabilities
    pi: list[N] first state probabilities
    O: list[T] enumerated observation sequence
    """

    N = A.shape[0]
    T = len(O)
    pi = np.asarray(pi)
    alpha = np.zeros((N,T))

    # Initialise the first column of alpha; each ith row is the probability
    # of starting in state i and observing the first observation (B[0]) there.
    alpha[:, 0] = pi * B[:, O[0]]

    for t in range(1, T):
        # each row of A to be pairwise multiplied by the row vector of B_t
        # so each (i,j) in i_j_ot is the probability of moving from state i
        # to j and then observing O[t] at j.
        # shape: (N,N)
        i_j_ot = A * B[:, O[t]] 

        # in alpha[:, t-1], the ith row is the probability of being in 
        # state 0 at time t-1 given the observation sequence up to t-1. 
        # Thus, matmul with i_j_ot 
        alpha[:, t] = alpha[:, t-1] @ i_j_ot

    return alpha

def forward_prob(A, B, pi, O):
    return forward(A, B, pi, O)[:, len(O)-1].sum()

In [202]:
forward_prob(A,B,pi,O)

np.float64(0.028562000000000008)

# Viterbi
A dynamic programming algorithm to find an optimal solution to the problem:

Given as input a Hidden Markov model $\lambda=(A,B)$ and a sequence of observations $O=o_1,o_2,...o_T$, find the most probable sequence of states $Q=q_1q_2q_3...q_T$.

In [203]:
def viterbi(A, B, pi, O):
    """
    A: ndarray(N, N) state transition probabilities
    B: ndarray(N, M) state observation probabilities
    pi: list[N] first state probabilities
    O: list[T] enumerated observation sequence
    """

    N = A.shape[0]
    T = len(O)
    # using logarithms to prevent underflow from tiny probabilities
    # the bestpathprob is only really for relative comparison so
    # if we need a probability can exp outside
    pi = np.log(np.asarray(pi))
    A = np.log(A)
    B = np.log(B)

    # every v[j, t] is the maximum probability over all state sequences
    # from 1 to t-1 of achieving said state sequence and the observation 
    # sequence from 1 to t and being in state j at time t given the model.
    v = np.zeros((N,T))
    bt = np.zeros((N,T), dtype=int)

    v[:, 0] = pi + B[:, O[0]]
    bt[:, 0] = 0

    for t in range(1, T):
        i_j_ot = A + B[:, O[t]] # shape: (n,n)
        v_last = v[:, t-1].reshape(-1,1) # shape: (n,1)

        # each entry of v_last is the probability of the best path ending 
        # in that state at t-1, jointly with the observation sequence thus far
        # the reshape ensures that each (i, j) in v_last * i_j_ot
        # gives the probability of being in state j from state i
        # axis=0 takes the most likely previous state from destination
        v[:, t] = np.max(v_last + i_j_ot, axis=0)
        bt[:, t] = np.argmax(v_last + i_j_ot, axis=0)

    # most likely final state at time T
    bestpathpointer = np.argmax(v[:, T-1])
    bestpath = [None] * T
    bestpath[T-1] = bestpathpointer
    for i in range(1, T):
        # bt[s, t] is the most likely state at t-1 which led to state s at time t
        bestpath[T-1-i] = bt[bestpath[T-i], T-i]

    bestpathprobability = v[:, T-1].max()

    return [int(state) for state in bestpath], bestpathprobability

In [204]:
bp, bpp = viterbi(A, B, pi, O)
print(bp)
print(bpp)

[1, 0, 1]
-4.358310108056566


# Ocassionally Dishonest Casino
Implementation of the ocassionaly dishonest casino example from Durbin's Biological Sequence Analysis.

In [205]:
# Fair - Loaded
Q = np.array([[0.95, 0.05], [0.1, 0.9]])
B = np.array([[1/6, 1/6, 1/6, 1/6, 1/6, 1/6], [1/10, 1/10, 1/10, 1/10, 1/10, 1/2]])
pi = [1/2, 1/2]

Generate 300 long observation sequence

In [206]:
def sample(probs):
    return np.random.choice(len(probs), p=probs)

O = [] # observation sequence
D = [] # actual state sequence

s = sample(pi) # initial die
for i in range(300):
    D.append(s)
    die_probs = B[s]
    O.append(sample(die_probs))
    s = sample(Q[s]) # transition to next state

In [207]:
def stagger_printing(bp, D, O):
    m = len(bp)
    c = 0
    while c < m:
        if m - c > 20:
            print("viterbi: " + str(bp[c:c+20]))
            print("actual: " + str(D[c:c+20]))
            print("die: " + str(O[c:c+20]))
        else:
            print("viterbi: " + str(bp[c:]))
            print("actual: " + str(D[c:]))
            print("die: " + str(O[c:]))
        c+=20
        print()

In [208]:
bp, bpp = viterbi(Q, B, pi, O)
stagger_printing(bp, D, O)
bpp

viterbi: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
actual: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
die: [5, 0, 0, 5, 5, 4, 1, 5, 5, 5, 5, 5, 5, 5, 4, 0, 5, 5, 3, 2]

viterbi: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
actual: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
die: [3, 5, 1, 3, 5, 5, 4, 1, 0, 4, 5, 5, 2, 5, 4, 5, 2, 4, 1, 1]

viterbi: [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
actual: [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
die: [5, 5, 5, 0, 1, 1, 2, 3, 2, 1, 1, 2, 2, 0, 3, 5, 2, 0, 4, 5]

viterbi: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
actual: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
die: [4, 5, 5, 1, 5, 3, 5, 1, 2, 3, 4, 2, 5, 3, 4, 5, 2, 3, 4, 1]

viterbi: [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
actual: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
die: [2, 2, 5, 1, 5, 3, 5, 2, 5, 5, 2

np.float64(-541.5031891218092)

Viterbi does pretty bad here. Mathematically, viterbi will always compute the most probable state sequence. I will calculate the probability of achieving the actual sequence and the dice rolls. Viterbi's probability should be higher.

In [209]:
def path_prob(A, B, pi, path, O):
    transitions = A[path[:-1], path[1:]]
    emissions = B[path, O]
    return np.log(pi[path[0]]) + np.log(emissions[0]) + np.sum([np.log(transitions), np.log(emissions[1:])])

In [210]:
path_prob(Q, B, pi, D, O)

np.float64(-585.2628678821625)

It is in fact less!

Let's do one last check, does this path_prob with viterbi's path produce the same result as the viterbi function?

In [211]:
path_prob(Q, B, pi, bp, O)

np.float64(-541.5031891218076)

:)